<a href="https://colab.research.google.com/github/aidakamalova-dev/olist-customer-satisfaction/blob/main/01_db_and_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Устанавливаем библиотеку kagglehub
!pip install kagglehub -q

import kagglehub
import pandas as pd
import sqlite3
import os
import glob

In [2]:
# 1. Скачиваем датасет
print("Начинаю скачивание датасета...")
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print(f"Данные успешно скачаны в папку: {path}\n")

Начинаю скачивание датасета...
Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Данные успешно скачаны в папку: /kaggle/input/brazilian-ecommerce



In [3]:
# 2. Создаем базу данных SQLite и подключаемся к ней
db_path = 'olist.db'
conn = sqlite3.connect(db_path)
print("База данных olist.db создана.\n")

База данных olist.db создана.



In [4]:
# 3. Находим все CSV-файлы в скачанной папке
csv_files = glob.glob(os.path.join(path, "*.csv"))

In [6]:
# 4. В цикле читаем каждый CSV и загружаем его как таблицу в SQLite
print("Начинаю загрузку таблиц в базу данных...")
for file_path in csv_files:
    # Получаем имя файла без расширения
    table_name = os.path.basename(file_path).replace('.csv', '')

    # Читаем CSV с помощью pandas
    df = pd.read_csv(file_path)

    # Записываем датафрейм в таблицу SQLite
    df.to_sql(table_name, conn, if_exists='replace', index=False)

    print(f"✅ Таблица {table_name} загружена (строк: {len(df)})")

Начинаю загрузку таблиц в базу данных...
✅ Таблица olist_customers_dataset загружена (строк: 99441)
✅ Таблица olist_sellers_dataset загружена (строк: 3095)
✅ Таблица olist_order_reviews_dataset загружена (строк: 99224)
✅ Таблица olist_order_items_dataset загружена (строк: 112650)
✅ Таблица olist_products_dataset загружена (строк: 32951)
✅ Таблица olist_geolocation_dataset загружена (строк: 1000163)
✅ Таблица product_category_name_translation загружена (строк: 71)
✅ Таблица olist_orders_dataset загружена (строк: 99441)
✅ Таблица olist_order_payments_dataset загружена (строк: 103886)


In [7]:
# 5. Проверяем, какие таблицы появились в базе
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("\nСписок таблиц, доступных теперь в твоей БД:")
for table in tables:
    print(f"- {table[0]}")


Список таблиц, доступных теперь в твоей БД:
- olist_customers_dataset
- olist_sellers_dataset
- olist_order_reviews_dataset
- olist_order_items_dataset
- olist_products_dataset
- olist_geolocation_dataset
- product_category_name_translation
- olist_orders_dataset
- olist_order_payments_dataset


In [8]:
# Закрываем соединение с базой на всякий случай
conn.close()

In [9]:
import sqlite3
import pandas as pd

# Подключаемся к базе SQLite
conn = sqlite3.connect('olist.db')

# SQL-запрос для объединения таблиц и расчета первичных признаков
query = """
SELECT
    o.order_id,
    o.customer_id,
    r.review_score,

    -- Сроки доставки в днях (ключевой бизнес-фактор)
    ROUND(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp), 2) AS delivery_time_days,
    ROUND(julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date), 2) AS delivery_delay_days,

    -- Агрегированные метрики по товарам в заказе
    i.total_items,
    i.total_price,
    i.total_freight,

    -- Агрегированные метрики по платежам
    p.total_payment_value,
    p.max_installments,

    -- Характеристики первого/основного товара
    pr.product_category_name,
    pr.product_weight_g

FROM olist_orders_dataset o

-- Присоединяем отзывы (наша целевая переменная)
INNER JOIN olist_order_reviews_dataset r
    ON o.order_id = r.order_id

-- Присоединяем агрегированные товарные позиции
LEFT JOIN (
    SELECT
        order_id,
        COUNT(order_item_id) AS total_items,
        SUM(price) AS total_price,
        SUM(freight_value) AS total_freight,
        MIN(product_id) AS main_product_id
    FROM olist_order_items_dataset
    GROUP BY order_id
) i ON o.order_id = i.order_id

-- Присоединяем агрегированные платежи
LEFT JOIN (
    SELECT
        order_id,
        SUM(payment_value) AS total_payment_value,
        MAX(payment_installments) AS max_installments
    FROM olist_order_payments_dataset
    GROUP BY order_id
) p ON o.order_id = p.order_id

-- Присоединяем параметры товара
LEFT JOIN olist_products_dataset pr
    ON i.main_product_id = pr.product_id

WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL;
"""

# Загружаем результат SQL-запроса в pandas DataFrame
df_features = pd.read_sql_query(query, conn)

# Закрываем соединение с БД
conn.close()

# Выводим информацию и первые 5 строк полученной витрины
print(f"Витрина успешно собрана! Размер: {df_features.shape[0]} строк, {df_features.shape[1]} колонок.\n")
print(df_features.head())

Витрина успешно собрана! Размер: 96353 строк, 12 колонок.

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

   review_score  delivery_time_days  delivery_delay_days  total_items  \
0             4                8.44                -7.11            1   
1             4               13.78                -5.36            1   
2             5                9.39               -17.25            1   
3             5               13.21               -12.98            1   
4             5                2.87                -9.24            1   

   total_price  total_freight  total_payment_value  max_install